# Feature Engineering

In [1]:
#%pip install -r ../requirements.txt -q

In [37]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [38]:
import pandas as pd
import numpy as np
import os

split_dir = "../data/splits"
train = pd.read_csv(os.path.join(split_dir, "train.csv"))
val   = pd.read_csv(os.path.join(split_dir, "val.csv"))
test  = pd.read_csv(os.path.join(split_dir, "test.csv"))

print("Loaded splits successfully:")
print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)


Loaded splits successfully:
Train: (4453834, 11)
Val: (954393, 11)
Test: (954393, 11)


### Feature- amount_to_oldbalanceOrg

In [39]:
def add_amount_ratio(df):
    df['amount_to_oldbalanceOrg'] = df['amount'] / (df['oldbalanceOrg'] + 1e-9)
    return df

train = add_amount_ratio(train)
val   = add_amount_ratio(val)
test  = add_amount_ratio(test)

print("✅ Added 'amount_to_oldbalanceOrg' to all splits")


✅ Added 'amount_to_oldbalanceOrg' to all splits


Drop unusable columns due to Kaggle rule

In [40]:
# === Drop unwanted balance columns ===
cols_to_drop = ['newbalanceOrig', 'newbalanceDest', 'oldbalanceDest']
for df in [train, val, test]:
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("Remaining columns (train):", train.columns.tolist())

Remaining columns (train): ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg']


### Receiver Statistics Features- avgAmountToDest, maxAmountReceived, stdAmountReceived, std_to_mean_ratio

In [41]:
def add_avg_amount_to_dest(df):
    avg_amount_to_dest = (
        df.groupby(['nameOrig', 'nameDest'])['amount']
          .mean()
          .reset_index(name='avgAmountToDest')
    )
    df = df.merge(avg_amount_to_dest, on=['nameOrig', 'nameDest'], how='left')
    return df

# Apply separately to avoid data leakage
train = add_avg_amount_to_dest(train)
val   = add_avg_amount_to_dest(val)
test  = add_avg_amount_to_dest(test)


print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest']
Test columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest']


In [42]:
def add_receiver_amount_stats(df):
    # Calculate receiver stats
    receiver_amt_stats = (
        df.groupby('nameDest')['amount']
          .agg(['max', 'std'])
          .reset_index()
          .rename(columns={'max': 'maxAmountReceived', 'std': 'stdAmountReceived'})
    )

    # Merge back
    df = df.merge(receiver_amt_stats, on='nameDest', how='left')

    # Add std-to-mean ratio using your earlier avgAmountToDest
    df['std_to_mean_ratio'] = df['stdAmountReceived'] / (df['avgAmountToDest'] + 1e-9)
    return df


# === Apply safely per split ===
train = add_receiver_amount_stats(train)
val   = add_receiver_amount_stats(val)
test  = add_receiver_amount_stats(test)

print("✅ Added receiver stats (maxAmountReceived, stdAmountReceived, std_to_mean_ratio)")

print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added receiver stats (maxAmountReceived, stdAmountReceived, std_to_mean_ratio)
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio']
Test columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio']


### Type-Based Amount Features- avgAmountPerType, p95AmountPerType, amountTypeRatio, typeHighValueFlag

In [47]:
def add_type_amount_features(df):
    type_stats = df.groupby('type')['amount'].agg(
        avgAmountPerType='mean',
        p95AmountPerType=lambda x: np.percentile(x, 95)
    ).reset_index()
    df = df.merge(type_stats, on='type', how='left')
    df['amountTypeRatio'] = df['amount'] / (df['avgAmountPerType'] + 1e-6)
    df['typeHighValueFlag'] = (df['amount'] > df['p95AmountPerType']).astype(int)
    return df

# Apply separately to avoid data leakage
train = add_type_amount_features(train)
val   = add_type_amount_features(val)
test  = add_type_amount_features(test)

print("✅ Added type amount features")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added type amount features
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag']
Test columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag']


### Feature - amountLog

In [48]:
def add_amount_features(df):
    # Log-transform to reduce skew
    df['amountLog'] = np.log1p(df['amount'])
    return df

# Apply per split
train = add_amount_features(train)
val   = add_amount_features(val)
test  = add_amount_features(test)

print("✅ Added 'amountLog' safely to all splits")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added 'amountLog' safely to all splits
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog']
Test columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog']


### Sender Stats Features - totalSent, meanSent, stdSent, numSent, fraction_CASH_IN, fraction_CASH_OUT, fraction_DEBIT, fraction_PAYMENT, fraction_TRANSFER

In [49]:
def add_sender_features(df):
    # Aggregate amount statistics per sender
    sender_agg = (
        df.groupby('nameOrig')['amount']
          .agg(['sum', 'mean', 'std', 'count'])
          .reset_index()
          .rename(columns={
              'sum': 'totalSent',
              'mean': 'meanSent',
              'std': 'stdSent',
              'count': 'numSent'
          })
    )

    # Fraction of each transaction type per sender
    type_counts = df.groupby(['nameOrig', 'type']).size().unstack(fill_value=0)
    type_fractions = type_counts.div(type_counts.sum(axis=1), axis=0).reset_index()
    type_fractions.columns = ['nameOrig'] + [f'fraction_{col}' for col in type_fractions.columns[1:]]

    # Merge both together
    sender_agg = sender_agg.merge(type_fractions, on='nameOrig', how='left')

    return sender_agg

# === Apply per split ===
train_sender = add_sender_features(train)
val_sender   = add_sender_features(val)
test_sender  = add_sender_features(test)

# Merge these aggregates back to main splits
train = train.merge(train_sender, on='nameOrig', how='left')
val   = val.merge(val_sender, on='nameOrig', how='left')
test  = test.merge(test_sender, on='nameOrig', how='left')

print("✅ Added sender aggregation features (no leakage)")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added sender aggregation features (no leakage)
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER']
Test columns: ['step', 'type', 

### Receiver Stats Features - totalReceived, meanReceived, stdReceived, numReceived,fraction_recv_CASH_IN, fraction_recv_CASH_OUT, fraction_recv_DEBIT, fraction_recv_PAYMENT, fraction_recv_TRANSFER

In [50]:
def add_receiver_features(df):
    # Aggregate amount statistics per receiver
    receiver_agg = (
        df.groupby('nameDest')['amount']
          .agg(['sum', 'mean', 'std', 'count'])
          .reset_index()
          .rename(columns={
              'sum': 'totalReceived',
              'mean': 'meanReceived',
              'std': 'stdReceived',
              'count': 'numReceived'
          })
    )

    # Fraction of each transaction type per receiver
    type_counts_recv = df.groupby(['nameDest', 'type']).size().unstack(fill_value=0)
    type_fractions_recv = type_counts_recv.div(type_counts_recv.sum(axis=1), axis=0).reset_index()
    type_fractions_recv.columns = ['nameDest'] + [f'fraction_recv_{col}' for col in type_fractions_recv.columns[1:]]

    # Merge both together
    receiver_agg = receiver_agg.merge(type_fractions_recv, on='nameDest', how='left')

    return receiver_agg

# === Apply per split ===
train_recv = add_receiver_features(train)
val_recv   = add_receiver_features(val)
test_recv  = add_receiver_features(test)

# Merge these aggregates back to each split
train = train.merge(train_recv, on='nameDest', how='left')
val   = val.merge(val_recv, on='nameDest', how='left')
test  = test.merge(test_recv, on='nameDest', how='left')

print("✅ Added receiver aggregation features (no leakage)")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added receiver aggregation features (no leakage)
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', '

### Neighbour Fraud Ratio Features- fraudRatioAmongReceivers, fraudRatioAmongSenders

In [51]:
def add_neighbor_fraud_features(df):
    # For each sender: how many of their receivers were involved in fraud (within same split)
    receiver_fraud = df.groupby('nameDest')['isFraud'].max().reset_index().rename(columns={'isFraud': 'receiver_is_fraud'})
    df = df.merge(receiver_fraud, on='nameDest', how='left')

    sender_neighbor_fraud = (
        df.groupby('nameOrig')['receiver_is_fraud']
          .agg(['mean'])
          .reset_index()
          .rename(columns={'mean': 'fraudRatioAmongReceivers'})
    )

    # For each receiver: how many of their senders were involved in fraud (within same split)
    sender_fraud = df.groupby('nameOrig')['isFraud'].max().reset_index().rename(columns={'isFraud': 'sender_is_fraud'})
    df = df.merge(sender_fraud, on='nameOrig', how='left')

    receiver_neighbor_fraud = (
        df.groupby('nameDest')['sender_is_fraud']
          .agg(['mean'])
          .reset_index()
          .rename(columns={'mean': 'fraudRatioAmongSenders'})
    )

    # Merge both summaries back to original df
    df = df.merge(sender_neighbor_fraud, on='nameOrig', how='left')
    df = df.merge(receiver_neighbor_fraud, on='nameDest', how='left')

    # Cleanup helper columns
    df.drop(columns=['receiver_is_fraud', 'sender_is_fraud'], inplace=True, errors='ignore')

    return df

# Apply per split
train = add_neighbor_fraud_features(train)
val   = add_neighbor_fraud_features(val)
test  = add_neighbor_fraud_features(test)

print("✅ Added leak-free neighbor fraud ratio features.")
print("Train columns:", train.columns.tolist())


✅ Added leak-free neighbor fraud ratio features.
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders']


### Time-based Features - hourOfDay, dayOfWeek, dayOfWeekName

In [52]:
def add_temporal_features(df):
    df['hourOfDay'] = df['step'] % 24
    df['dayOfWeek'] = (df['step'] // 24) % 7
    day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    df['dayOfWeekName'] = df['dayOfWeek'].map(lambda x: day_labels[int(x)])
    return df

# Apply per split
train = add_temporal_features(train)
val   = add_temporal_features(val)
test  = add_temporal_features(test)

print("✅ Added temporal features: hourOfDay, dayOfWeek, dayOfWeekName")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added temporal features: hourOfDay, dayOfWeek, dayOfWeekName
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountRece

### In-Degree and Out-Degree features- numUniqueDest, numUniqueOrig

In [53]:
def add_degree_features(df):
    # Number of unique receivers each sender has
    num_unique_dest = (
        df.groupby('nameOrig')['nameDest']
          .nunique()
          .reset_index()
          .rename(columns={'nameDest': 'numUniqueDest'})
    )

    # Number of unique senders each receiver has
    num_unique_orig = (
        df.groupby('nameDest')['nameOrig']
          .nunique()
          .reset_index()
          .rename(columns={'nameOrig': 'numUniqueOrig'})
    )

    # Merge both sets
    df = df.merge(num_unique_dest, on='nameOrig', how='left')
    df = df.merge(num_unique_orig, on='nameDest', how='left')

    # Replace NaNs with 0 (for accounts that only send or only receive)
    df[['numUniqueDest', 'numUniqueOrig']] = df[['numUniqueDest', 'numUniqueOrig']].fillna(0)

    return df

# === Apply per split ===
train = add_degree_features(train)
val   = add_degree_features(val)
test  = add_degree_features(test)

print("✅ Added degree features: numUniqueDest, numUniqueOrig")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added degree features: numUniqueDest, numUniqueOrig
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmount

### Transanction Recency Feature - transactionRecency

In [54]:
def add_transaction_recency(df):
    df = df.sort_values(by=['nameOrig', 'step']).reset_index(drop=True)
    df['transactionRecency'] = df.groupby('nameOrig')['step'].diff()
    df['transactionRecency'] = df['transactionRecency'].fillna(df['step'])
    return df

# === Apply to all splits ===
train = add_transaction_recency(train)
val   = add_transaction_recency(val)
test  = add_transaction_recency(test)

print("✅ Added transaction recency feature.")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added transaction recency feature.
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig', 'transactionRecency']
Val columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxA

### Transaction Type Sequence Features- prev_type_1, prev_type_2, prev_type_3, is_cashin_transfer, is_transfer_cashout

In [55]:
def add_transaction_sequence_features(df, n_last_transactions=3):
    df = df.sort_values(by=['nameOrig', 'step']).reset_index(drop=True)

    # Create lagged transaction types
    for i in range(1, n_last_transactions + 1):
        df[f'prev_type_{i}'] = df.groupby('nameOrig')['type'].shift(i)

    # Simple flag: CASH-IN followed by TRANSFER → potential laundering pattern
    df['is_cashin_transfer'] = (
        (df['prev_type_1'] == 'CASH_IN') & (df['type'] == 'TRANSFER')
    ).astype(int)

    # Simple flag: TRANSFER followed by CASH-OUT → typical fraud exit pattern
    df['is_transfer_cashout'] = (
        (df['prev_type_1'] == 'TRANSFER') & (df['type'] == 'CASH_OUT')
    ).astype(int)

    return df

# === Apply per split ===
train = add_transaction_sequence_features(train)
val   = add_transaction_sequence_features(val)
test  = add_transaction_sequence_features(test)

print("✅ Added sequence features (is_cashin_transfer, is_transfer_cashout)")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())

✅ Added sequence features (is_cashin_transfer, is_transfer_cashout)
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig', 'transactionRecency', 'prev_type_1', 'prev_type_2', 'prev_type_3', 'is_cashin_transfer', 'is_transfer_cashout']
Val columns: ['step', 'type', 'amount', 

### % of received funds forwarded within 24h feature - pctForwarded

In [56]:
def add_forwarding_ratio(df):
    # For each account, total sent and received
    totals = df.groupby('nameOrig')['amount'].sum().reset_index(name='totalSent')
    received = df.groupby('nameDest')['amount'].sum().reset_index(name='totalReceived')

    merged = totals.merge(received, left_on='nameOrig', right_on='nameDest', how='outer').fillna(0)
    merged['pctForwarded'] = (merged['totalSent'] / (merged['totalReceived'] + 1e-9)) * 100

    df = df.merge(merged[['nameOrig', 'pctForwarded']], on='nameOrig', how='left')
    return df

# Apply per split
train = add_forwarding_ratio(train)
val   = add_forwarding_ratio(val)
test  = add_forwarding_ratio(test)

print("✅ Added simple 'pctForwarded' feature (no time window, no leakage).")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added simple 'pctForwarded' feature (no time window, no leakage).
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig', 'transactionRecency', 'prev_type_1', 'prev_type_2', 'prev_type_3', 'is_cashin_transfer', 'is_transfer_cashout', 'pctForwarded']
Val columns: ['step', 't

### Frequency of repeated sender–receiver pair Feature - pairFrequency

In [57]:
def add_pair_frequency(df):
    pair_frequency = (
        df.groupby(['nameOrig', 'nameDest'])
          .size()
          .reset_index(name='pairFrequency')
    )
    df = df.merge(pair_frequency, on=['nameOrig', 'nameDest'], how='left')
    return df

# === Apply per split ===
train = add_pair_frequency(train)
val   = add_pair_frequency(val)
test  = add_pair_frequency(test)


print("✅ Added 'pairFrequency' feature safely per split.")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added 'pairFrequency' feature safely per split.
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig', 'transactionRecency', 'prev_type_1', 'prev_type_2', 'prev_type_3', 'is_cashin_transfer', 'is_transfer_cashout', 'pctForwarded', 'pairFrequency']
Val columns: ['step', 'ty

Since there are no repeated sender-receiver pairs, identify % of transactions from unique accounts

In [58]:
def add_unique_partner_features(df):
    # Unique receivers per sender
    unique_dest = df.groupby('nameOrig')['nameDest'].nunique().reset_index(name='numUniqueDest')
    total_sent = df.groupby('nameOrig').size().reset_index(name='totalSent')
    sender_stats = unique_dest.merge(total_sent, on='nameOrig', how='left')
    sender_stats['pctUniqueDest'] = (sender_stats['numUniqueDest'] / sender_stats['totalSent']) * 100

    # Unique senders per receiver
    unique_orig = df.groupby('nameDest')['nameOrig'].nunique().reset_index(name='numUniqueOrig')
    total_received = df.groupby('nameDest').size().reset_index(name='totalReceived')
    receiver_stats = unique_orig.merge(total_received, on='nameDest', how='left')
    receiver_stats['pctUniqueOrig'] = (receiver_stats['numUniqueOrig'] / receiver_stats['totalReceived']) * 100

    # Merge back
    df = df.merge(sender_stats[['nameOrig', 'pctUniqueDest']], on='nameOrig', how='left')
    df = df.merge(receiver_stats[['nameDest', 'pctUniqueOrig']], on='nameDest', how='left')

    # Fill NaNs for accounts that only send or only receive
    df[['pctUniqueDest', 'pctUniqueOrig']] = df[['pctUniqueDest', 'pctUniqueOrig']].fillna(0)
    return df


# === Apply per split ===
train = add_unique_partner_features(train)
val   = add_unique_partner_features(val)
test  = add_unique_partner_features(test)

print("✅ Added unique partner ratio features: pctUniqueDest, pctUniqueOrig")
print("Train columns:", train.columns.tolist())
print("Val columns:", val.columns.tolist())
print("Test columns:", test.columns.tolist())


✅ Added unique partner ratio features: pctUniqueDest, pctUniqueOrig
Train columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'nameDest', 'isFlaggedFraud', 'isFraud', 'amount_to_oldbalanceOrg', 'avgAmountToDest', 'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 'avgAmountPerType', 'p95AmountPerType', 'amountTypeRatio', 'typeHighValueFlag', 'amountLog', 'totalSent', 'meanSent', 'stdSent', 'numSent', 'fraction_CASH_IN', 'fraction_CASH_OUT', 'fraction_DEBIT', 'fraction_PAYMENT', 'fraction_TRANSFER', 'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'fraction_recv_CASH_IN', 'fraction_recv_CASH_OUT', 'fraction_recv_DEBIT', 'fraction_recv_PAYMENT', 'fraction_recv_TRANSFER', 'fraudRatioAmongReceivers', 'fraudRatioAmongSenders', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'numUniqueDest', 'numUniqueOrig', 'transactionRecency', 'prev_type_1', 'prev_type_2', 'prev_type_3', 'is_cashin_transfer', 'is_transfer_cashout', 'pctForwarded', 'pairFrequency', 'pctUni